# **Data Download From Open Images**

This whole Block Of Code is to download the data from Open Image v6 Website. If you want to explore the dataset you can visit https://storage.googleapis.com/openimages/web/index.html . Here you can find lots of object classes you can pick any number of classes you want and download the images and its corresponding annotations. For object detection/ Segmentation.Here I am refering https://github.com/EscVM/OIDv4_ToolKit this repository just to download the data from Open Image v6 since its very well written.

In [ ]:
"""This repository gives best source code to dowanload the data from Open Image v6"""
!git clone https://github.com/EscVM/OIDv4_ToolKit.git

In [ ]:
import pandas
import numpy
import awscli
import urllib3
import tqdm
import cv2

In [ ]:
import os
os.chdir('/content/OIDv4_ToolKit')

In [ ]:
import modules

In [ ]:
!python3 main.py -h

In [ ]:
"""
  Classes - You can choose any number of classes here simply put the class name after refering the Open Image available classes
  type- Mention train/Vlidation/test
  Limit - Mention the number of images per classyou want
  -- y/n for command line yes/no arguments. Keep it y
""" 
!python3 main.py downloader --classes Car Bus Truck --type_csv train --limit 1000 --y

In [ ]:
!python3 main.py downloader --classes Car Bus Truck --type_csv validation --limit 200 --y

In [ ]:
 !python3 main.py downloader --classes Car Bus Truck --type_csv test --limit 10 --y

In [ ]:
os.chdir('/content')

# **Creating a TFRecord**

This whole block of code is used to convert the data into TfrRecords which can be used for training or validation

In [ ]:
import os
from tqdm import tqdm
from sys import exit
import argparse
import cv2
from textwrap import dedent
from lxml import etree

foldername = os.path.basename(os.getcwd())
if foldername == "tools": os.chdir("..")

Dataset_path = "/content/OIDv4_ToolKit/OID/Dataset"

def convert_to_xml():
    current_path = os.getcwd()
    os.chdir(Dataset_path)
    DIRS = os.listdir(os.getcwd())

    for DIR in DIRS:
        if os.path.isdir(DIR):
            os.chdir(DIR)

            print("Currently in Subdirectory:", DIR)
            CLASS_DIRS = os.listdir(os.getcwd()) 
            for CLASS_DIR in CLASS_DIRS:
                if " " in CLASS_DIR:
                    os.rename(CLASS_DIR, CLASS_DIR.replace(" ", "_"))
            
            CLASS_DIRS = os.listdir(os.getcwd())
            for CLASS_DIR in CLASS_DIRS:
                if os.path.isdir(CLASS_DIR):
                    os.chdir(CLASS_DIR)

                    print("\n" + "Creating PASCAL VOC XML Files for Class:", CLASS_DIR)
                    # Create Directory for annotations if it does not exist yet

                    #Read Labels from OIDv4 ToolKit
                    os.chdir("Label")

                    #Create PASCAL XML
                    for filename in tqdm(os.listdir(os.getcwd())):
                        if filename.endswith(".txt"):
                            filename_str = str.split(filename, ".")[0]


                            annotation = etree.Element("annotation")
                            
                            os.chdir("..")
                            folder = etree.Element("folder")
                            folder.text = os.path.basename(os.getcwd())
                            annotation.append(folder)

                            filename_xml = etree.Element("filename")
                            filename_xml.text = filename_str + ".jpg"
                            annotation.append(filename_xml)

                            path = etree.Element("path")
                            path.text = os.path.join(os.path.dirname(os.path.abspath(filename)), filename_str + ".jpg")
                            annotation.append(path)

                            source = etree.Element("source")
                            annotation.append(source)

                            database = etree.Element("database")
                            database.text = "Unknown"
                            source.append(database)

                            size = etree.Element("size")
                            annotation.append(size)

                            width = etree.Element("width")
                            height = etree.Element("height")
                            depth = etree.Element("depth")

                            img = cv2.imread(filename_xml.text)

                            try:
                                width.text = str(img.shape[1])
                            except AttributeError:
                                os.chdir("Label")
                                continue
                            height.text = str(img.shape[0])
                            depth.text = str(img.shape[2])

                            size.append(width)
                            size.append(height)
                            size.append(depth)

                            segmented = etree.Element("segmented")
                            segmented.text = "0"
                            annotation.append(segmented)

                            os.chdir("Label")
                            label_original = open(filename, 'r')

                            # Labels from OIDv4 Toolkit: name_of_class X_min Y_min X_max Y_max
                            for line in label_original:
                                line = line.strip()
                                l = line.split(' ')
                                
                                class_name_len = len(l) - 4 # 4 coordinates
                                class_name = l[0]
                                for i in range(1,class_name_len):
                                    class_name = f"{class_name}_{l[i]}"

                                addi = class_name_len

                                xmin_l = str(int(round(float(l[0+addi]))))
                                ymin_l = str(int(round(float(l[1+addi]))))
                                xmax_l = str(int(round(float(l[2+addi]))))
                                ymax_l = str(int(round(float(l[3+addi]))))
                                
                                obj = etree.Element("object")
                                annotation.append(obj)

                                name = etree.Element("name")
                                name.text = class_name
                                obj.append(name)

                                pose = etree.Element("pose")
                                pose.text = "Unspecified"
                                obj.append(pose)

                                truncated = etree.Element("truncated")
                                truncated.text = "0"
                                obj.append(truncated)

                                difficult = etree.Element("difficult")
                                difficult.text = "0"
                                obj.append(difficult)

                                bndbox = etree.Element("bndbox")
                                obj.append(bndbox)

                                xmin = etree.Element("xmin")
                                xmin.text = xmin_l
                                bndbox.append(xmin)

                                ymin = etree.Element("ymin")
                                ymin.text = ymin_l
                                bndbox.append(ymin)

                                xmax = etree.Element("xmax")
                                xmax.text = xmax_l
                                bndbox.append(xmax)

                                ymax = etree.Element("ymax")
                                ymax.text = ymax_l
                                bndbox.append(ymax)
                            os.chdir("..")
                            # write xml to file
                            s = etree.tostring(annotation, pretty_print=True)
                            with open(filename_str + ".xml", 'wb') as f:
                                f.write(s)
                                f.close()
                            os.chdir("Label")
                    os.chdir("..")
                    os.chdir("..")       
            os.chdir("..")
convert_to_xml()


In [ ]:
import time
import os
import hashlib
from absl import app, flags, logging
from absl.flags import FLAGS
import tensorflow as tf
import lxml.etree
import tqdm

In [ ]:
def parse_xml(xml):
    if not len(xml):
        return {xml.tag: xml.text}
    result = {}
    for child in xml:
        child_result = parse_xml(child)
        if child.tag != 'object':
            result[child.tag] = child_result[child.tag]
        else:
            if child.tag not in result:
                result[child.tag] = []
            result[child.tag].append(child_result[child.tag])
    return {xml.tag: result}


In [ ]:
def build_example(img,data):
  
  xmin = []
  ymin = []
  xmax = []
  ymax = []
  classes = []
  classes_text = []
  truncated = []
  views = []
  difficult_obj = []
  width = int(data['annotation']['size']['width'])
  height = int(data['annotation']['size']['height'])
  for i in range(len(data['annotation']['object'])):

    xmin.append(int(data['annotation']['object'][i]['bndbox']['xmin'])/int(data['annotation']['size']['width']))
    ymin.append(int(data['annotation']['object'][i]['bndbox']['ymin'])/int(data['annotation']['size']['height']))
    xmax.append(int(data['annotation']['object'][i]['bndbox']['xmax'])/int(data['annotation']['size']['width']))
    ymax.append(int(data['annotation']['object'][i]['bndbox']['ymax'])/int(data['annotation']['size']['height']))
    classes_text.append(data['annotation']['object'][i]['name'].encode('utf8'))
    truncated.append(int(data['annotation']['object'][i]['truncated']))
    difficult_obj.append(int(data['annotation']['object'][i]['difficult']))
    views.append(data['annotation']['object'][i]['pose'].encode('utf8'))
    # print('#'*30)
  img_raw = open(img, 'rb').read()
  key = hashlib.sha256(img_raw).hexdigest()
  example = tf.train.Example(features=tf.train.Features(feature={
          'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
          'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
          'image/filename': tf.train.Feature(bytes_list=tf.train.BytesList(value=[
              data['annotation']['filename'].encode('utf8')])),
          'image/source_id': tf.train.Feature(bytes_list=tf.train.BytesList(value=[
              data['annotation']['filename'].encode('utf8')])),
          'image/key/sha256': tf.train.Feature(bytes_list=tf.train.BytesList(value=[key.encode('utf8')])),
          'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[img_raw])),
          'image/format': tf.train.Feature(bytes_list=tf.train.BytesList(value=['jpeg'.encode('utf8')])),
          'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmin)),
          'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmax)),
          'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymin)),
          'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymax)),
          'image/object/class/text': tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
          # 'image/object/class/label': tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
          'image/object/difficult': tf.train.Feature(int64_list=tf.train.Int64List(value=difficult_obj)),
          'image/object/truncated': tf.train.Feature(int64_list=tf.train.Int64List(value=truncated)),
          'image/object/view': tf.train.Feature(bytes_list=tf.train.BytesList(value=views)),
      }))
  return example


In [ ]:
path = '/content/OIDv4_ToolKit/OID/Dataset/'
writer = tf.io.TFRecordWriter('/content/data/OID_train.tfrecord')

for j in sorted(os.listdir(path+'train'+'/')):
    for k in sorted(os.listdir(path+'train'+'/'+j+'/')):
      dir = (path+'train'+'/'+j+'/'+k)
      if (dir.split('.')[-1])=='jpg':
        img = dir
      if (dir.split('.')[-1])=='xml':
        data = parse_xml(lxml.etree.fromstring(open(dir).read()))
        tf_example = build_example(img,data)    
        writer.write(tf_example.SerializeToString())
writer.close()
logging.info("Done")

In [ ]:
path = '/content/OIDv4_ToolKit/OID/Dataset/'
writer = tf.io.TFRecordWriter('/content/data/OID_validation.tfrecord')

for j in sorted(os.listdir(path+'validation'+'/')):
    for k in sorted(os.listdir(path+'validation'+'/'+j+'/')):
      dir = (path+'validation'+'/'+j+'/'+k)
      if (dir.split('.')[-1])=='jpg':
        img = dir
      if (dir.split('.')[-1])=='xml':
        data = parse_xml(lxml.etree.fromstring(open(dir).read()))
        tf_example = build_example(img,data)    
        writer.write(tf_example.SerializeToString())
writer.close()
logging.info("Done")

# **Training**

This Whole Block of code explains the training process for Object detection using YOLOv3 from scratch. Although my trained network performance was poor as compared to YOLOv3 oretrained network. I wanted to try the training rhe model from scratch so I tried this approach.

Utils

In [ ]:
from absl import logging
import numpy as np
import tensorflow as tf
import cv2

YOLOV3_LAYER_LIST = [
    'yolo_darknet',
    'yolo_conv_0',
    'yolo_output_0',
    'yolo_conv_1',
    'yolo_output_1',
    'yolo_conv_2',
    'yolo_output_2',
]

YOLOV3_TINY_LAYER_LIST = [
    'yolo_darknet',
    'yolo_conv_0',
    'yolo_output_0',
    'yolo_conv_1',
    'yolo_output_1',
]


def load_darknet_weights(model, weights_file, tiny=False):
    wf = open(weights_file, 'rb')
    major, minor, revision, seen, _ = np.fromfile(wf, dtype=np.int32, count=5)

    if tiny:
        layers = YOLOV3_TINY_LAYER_LIST
    else:
        layers = YOLOV3_LAYER_LIST

    for layer_name in layers:
        sub_model = model.get_layer(layer_name)
        for i, layer in enumerate(sub_model.layers):
            if not layer.name.startswith('conv2d'):
                continue
            batch_norm = None
            if i + 1 < len(sub_model.layers) and \
                    sub_model.layers[i + 1].name.startswith('batch_norm'):
                batch_norm = sub_model.layers[i + 1]

            logging.info("{}/{} {}".format(
                sub_model.name, layer.name, 'bn' if batch_norm else 'bias'))

            filters = layer.filters
            size = layer.kernel_size[0]
            in_dim = layer.get_input_shape_at(0)[-1]

            if batch_norm is None:
                conv_bias = np.fromfile(wf, dtype=np.float32, count=filters)
            else:
                # darknet [beta, gamma, mean, variance]
                bn_weights = np.fromfile(
                    wf, dtype=np.float32, count=4 * filters)
                # tf [gamma, beta, mean, variance]
                bn_weights = bn_weights.reshape((4, filters))[[1, 0, 2, 3]]

            # darknet shape (out_dim, in_dim, height, width)
            conv_shape = (filters, in_dim, size, size)
            conv_weights = np.fromfile(
                wf, dtype=np.float32, count=np.product(conv_shape))
            # tf shape (height, width, in_dim, out_dim)
            conv_weights = conv_weights.reshape(
                conv_shape).transpose([2, 3, 1, 0])

            if batch_norm is None:
                layer.set_weights([conv_weights, conv_bias])
            else:
                layer.set_weights([conv_weights])
                batch_norm.set_weights(bn_weights)

    assert len(wf.read()) == 0, 'failed to read all data'
    wf.close()


def broadcast_iou(box_1, box_2):
    # box_1: (..., (x1, y1, x2, y2))
    # box_2: (N, (x1, y1, x2, y2))

    # broadcast boxes
    box_1 = tf.expand_dims(box_1, -2)
    box_2 = tf.expand_dims(box_2, 0)
    # new_shape: (..., N, (x1, y1, x2, y2))
    new_shape = tf.broadcast_dynamic_shape(tf.shape(box_1), tf.shape(box_2))
    box_1 = tf.broadcast_to(box_1, new_shape)
    box_2 = tf.broadcast_to(box_2, new_shape)

    int_w = tf.maximum(tf.minimum(box_1[..., 2], box_2[..., 2]) -
                       tf.maximum(box_1[..., 0], box_2[..., 0]), 0)
    int_h = tf.maximum(tf.minimum(box_1[..., 3], box_2[..., 3]) -
                       tf.maximum(box_1[..., 1], box_2[..., 1]), 0)
    int_area = int_w * int_h
    box_1_area = (box_1[..., 2] - box_1[..., 0]) * \
        (box_1[..., 3] - box_1[..., 1])
    box_2_area = (box_2[..., 2] - box_2[..., 0]) * \
        (box_2[..., 3] - box_2[..., 1])
    return int_area / (box_1_area + box_2_area - int_area)


def draw_outputs(img, outputs, class_names):
    # boxes, objectness, classes, nums = outputs
    # print(boxes,objectness,classes,nums)
    boxes, objectness, classes, nums = outputs
    boxes, objectness, classes, nums = boxes[0], objectness[0], classes[0], nums[0]
    wh = np.flip(img.shape[0:2])
    # print('classes are:',classes)
    for i in range(nums):
        x1y1 = tuple((np.array(boxes[i][0:2]) * wh).astype(np.int32))
        x2y2 = tuple((np.array(boxes[i][2:4]) * wh).astype(np.int32))
        img = cv2.rectangle(img, x1y1, x2y2, (255, 0, 0), 2)
        # print(class_names[int(classes[i])])
        img = cv2.putText(img, '{} {:.4f}'.format(class_names[int(classes[i])], objectness[i]),
        # img = cv2.putText(img,'{}'.format(objectness[i]),
            x1y1, cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 0, 255), 2)
    return img
  
   


def draw_labels(x, y, class_names):
    img = x.numpy()
    boxes, classes = tf.split(y, (4, 1), axis=-1)
    classes = classes[..., 0]
    wh = np.flip(img.shape[0:2])
    for i in range(len(boxes)):
        x1y1 = tuple((np.array(boxes[i][0:2]) * wh).astype(np.int32))
        x2y2 = tuple((np.array(boxes[i][2:4]) * wh).astype(np.int32))
        img = cv2.rectangle(img, x1y1, x2y2, (255, 0, 0), 2)
        img = cv2.putText(img, class_names[classes[i]],
                          x1y1, cv2.FONT_HERSHEY_COMPLEX_SMALL,
                          1, (0, 0, 255), 2)
    return img


def freeze_all(model, frozen=True):
    model.trainable = not frozen
    if isinstance(model, tf.keras.Model):
        for l in model.layers:
            freeze_all(l, frozen)

def convert_boxes(image, boxes):
    returned_boxes = []
    for box in boxes:
        box[0] = (box[0] * image.shape[1]).astype(int)
        box[1] = (box[1] * image.shape[0]).astype(int)
        box[2] = (box[2] * image.shape[1]).astype(int)
        box[3] = (box[3] * image.shape[0]).astype(int)
        box[2] = int(box[2]-box[0])
        box[3] = int(box[3]-box[1])
        box = box.astype(int)
        box = box.tolist()
        if box != [0,0,0,0]:
            returned_boxes.append(box)
    return returned_boxes


Dataset

In [ ]:
import tensorflow as tf
from absl.flags import FLAGS

@tf.function
def transform_targets_for_output(y_true, grid_size, anchor_idxs):
    # y_true: (N, boxes, (x1, y1, x2, y2, class, best_anchor))
    N = tf.shape(y_true)[0]

    # y_true_out: (N, grid, grid, anchors, [x1, y1, x2, y2, obj, class])
    y_true_out = tf.zeros(
        (N, grid_size, grid_size, tf.shape(anchor_idxs)[0], 6))

    anchor_idxs = tf.cast(anchor_idxs, tf.int32)

    indexes = tf.TensorArray(tf.int32, 1, dynamic_size=True)
    updates = tf.TensorArray(tf.float32, 1, dynamic_size=True)
    idx = 0
    for i in tf.range(N):
        for j in tf.range(tf.shape(y_true)[1]):
            if tf.equal(y_true[i][j][2], 0):
                continue
            anchor_eq = tf.equal(
                anchor_idxs, tf.cast(y_true[i][j][5], tf.int32))

            if tf.reduce_any(anchor_eq):
                box = y_true[i][j][0:4]
                box_xy = (y_true[i][j][0:2] + y_true[i][j][2:4]) / 2

                anchor_idx = tf.cast(tf.where(anchor_eq), tf.int32)
                grid_xy = tf.cast(box_xy // (1/grid_size), tf.int32)

                # grid[y][x][anchor] = (tx, ty, bw, bh, obj, class)
                indexes = indexes.write(
                    idx, [i, grid_xy[1], grid_xy[0], anchor_idx[0][0]])
                updates = updates.write(
                    idx, [box[0], box[1], box[2], box[3], 1, y_true[i][j][4]])
                idx += 1

    # tf.print(indexes.stack())
    # tf.print(updates.stack())

    return tf.tensor_scatter_nd_update(
        y_true_out, indexes.stack(), updates.stack())


def transform_targets(y_train, anchors, anchor_masks, size):
    y_outs = []
    grid_size = size // 32

    # calculate anchor index for true boxes
    anchors = tf.cast(anchors, tf.float32)
    anchor_area = anchors[..., 0] * anchors[..., 1]
    box_wh = y_train[..., 2:4] - y_train[..., 0:2]
    box_wh = tf.tile(tf.expand_dims(box_wh, -2),
                     (1, 1, tf.shape(anchors)[0], 1))
    box_area = box_wh[..., 0] * box_wh[..., 1]
    intersection = tf.minimum(box_wh[..., 0], anchors[..., 0]) * \
        tf.minimum(box_wh[..., 1], anchors[..., 1])
    iou = intersection / (box_area + anchor_area - intersection)
    anchor_idx = tf.cast(tf.argmax(iou, axis=-1), tf.float32)
    anchor_idx = tf.expand_dims(anchor_idx, axis=-1)

    y_train = tf.concat([y_train, anchor_idx], axis=-1)

    for anchor_idxs in anchor_masks:
        y_outs.append(transform_targets_for_output(
            y_train, grid_size, anchor_idxs))
        grid_size *= 2

    return tuple(y_outs)


def transform_images(x_train, size):
    x_train = tf.image.resize(x_train, (size, size))
    x_train = x_train / 255
    return x_train


# https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/using_your_own_dataset.md#conversion-script-outline-conversion-script-outline
# Commented out fields are not required in our project
IMAGE_FEATURE_MAP = {

    'image/encoded': tf.io.FixedLenFeature([], tf.string),
    'image/object/bbox/xmin': tf.io.VarLenFeature(tf.float32),
    'image/object/bbox/ymin': tf.io.VarLenFeature(tf.float32),
    'image/object/bbox/xmax': tf.io.VarLenFeature(tf.float32),
    'image/object/bbox/ymax': tf.io.VarLenFeature(tf.float32),
    'image/object/class/text': tf.io.VarLenFeature(tf.string),
}


def parse_tfrecord(tfrecord, class_table, size):
    x = tf.io.parse_single_example(tfrecord, IMAGE_FEATURE_MAP)
    x_train = tf.image.decode_jpeg(x['image/encoded'], channels=3)
    x_train = tf.image.resize(x_train, (size, size))
    class_text = tf.sparse.to_dense(
        x['image/object/class/text'], default_value='')
    labels = tf.cast(class_table.lookup(class_text), tf.float32)
    y_train = tf.stack([tf.sparse.to_dense(x['image/object/bbox/xmin']),
                        tf.sparse.to_dense(x['image/object/bbox/ymin']),
                        tf.sparse.to_dense(x['image/object/bbox/xmax']),
                        tf.sparse.to_dense(x['image/object/bbox/ymax']),
                        labels], axis=1)

    paddings = [[0, 100 - tf.shape(y_train)[0]], [0, 0]]
    y_train = tf.pad(y_train, paddings)
    # print(y_train.shape)
    return x_train, y_train


def load_tfrecord_dataset(file_pattern, class_file, size=416):
    LINE_NUMBER = -1  # TODO: use tf.lookup.TextFileIndex.LINE_NUMBER
    class_table = tf.lookup.StaticHashTable(tf.lookup.TextFileInitializer(
        class_file, tf.string, 0, tf.int64, LINE_NUMBER, delimiter="\n"), -1)

    files = tf.data.Dataset.list_files(file_pattern)
    dataset = files.flat_map(tf.data.TFRecordDataset)
    return dataset.map(lambda x: parse_tfrecord(x, class_table, size))


def load_fake_dataset():
    x_train = tf.image.decode_jpeg(
        open('./data/girl.png', 'rb').read(), channels=3)
    x_train = tf.expand_dims(x_train, axis=0)

    labels = [
        [0.18494931, 0.03049111, 0.9435849,  0.96302897, 0],
        [0.01586703, 0.35938117, 0.17582396, 0.6069674, 56],
        [0.09158827, 0.48252046, 0.26967454, 0.6403017, 67]
    ] + [[0, 0, 0, 0, 0]] * 5
    y_train = tf.convert_to_tensor(labels, tf.float32)
    y_train = tf.expand_dims(y_train, axis=0)

    return tf.data.Dataset.from_tensor_slices((x_train, y_train))


Model Definition

In [ ]:
from absl import flags
from absl.flags import FLAGS
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.layers import Add,Concatenate, Conv2D, Input,Lambda, LeakyReLU,  MaxPool2D, UpSampling2D,ZeroPadding2D, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import binary_crossentropy,sparse_categorical_crossentropy

# from .utils import broadcast_iou

# flags.DEFINE_integer('yolo_max_boxes', 100,
#                      'maximum number of boxes per image')
# flags.DEFINE_float('yolo_iou_threshold', 0.5, 'iou threshold')
# flags.DEFINE_float('yolo_score_threshold', 0.6, 'score threshold')

yolo_anchors = np.array([(10, 13), (16, 30), (33, 23), (30, 61), (62, 45),
                         (59, 119), (116, 90), (156, 198), (373, 326)],
                        np.float32) / 416
yolo_anchor_masks = np.array([[6, 7, 8], [3, 4, 5], [0, 1, 2]])

yolo_tiny_anchors = np.array([(10, 14), (23, 27), (37, 58),
                              (81, 82), (135, 169),  (344, 319)],
                             np.float32) / 416
yolo_tiny_anchor_masks = np.array([[3, 4, 5], [0, 1, 2]])


def DarknetConv(x, filters, size, strides=1, batch_norm=True):
    if strides == 1:
        padding = 'same'
    else:
        x = ZeroPadding2D(((1, 0), (1, 0)))(x)  # top left half-padding
        padding = 'valid'
    x = Conv2D(filters=filters, kernel_size=size,
               strides=strides, padding=padding,
               use_bias=not batch_norm, kernel_regularizer=l2(0.0005))(x)
    if batch_norm:
        x = BatchNormalization()(x)
        x = LeakyReLU(alpha=0.1)(x)
    return x


def DarknetResidual(x, filters):
    prev = x
    x = DarknetConv(x, filters // 2, 1)
    x = DarknetConv(x, filters, 3)
    x = Add()([prev, x])
    return x


def DarknetBlock(x, filters, blocks):
    x = DarknetConv(x, filters, 3, strides=2)
    for _ in range(blocks):
        x = DarknetResidual(x, filters)
    return x


def Darknet(name=None):
    x = inputs = Input([None, None, 3])
    x = DarknetConv(x, 32, 3)
    x = DarknetBlock(x, 64, 1)
    x = DarknetBlock(x, 128, 2)  # skip connection
    x = x_36 = DarknetBlock(x, 256, 8)  # skip connection
    x = x_61 = DarknetBlock(x, 512, 8)
    x = DarknetBlock(x, 1024, 4)
    return tf.keras.Model(inputs, (x_36, x_61, x), name=name)


def DarknetTiny(name=None):
    x = inputs = Input([None, None, 3])
    x = DarknetConv(x, 16, 3)
    x = MaxPool2D(2, 2, 'same')(x)
    x = DarknetConv(x, 32, 3)
    x = MaxPool2D(2, 2, 'same')(x)
    x = DarknetConv(x, 64, 3)
    x = MaxPool2D(2, 2, 'same')(x)
    x = DarknetConv(x, 128, 3)
    x = MaxPool2D(2, 2, 'same')(x)
    x = x_8 = DarknetConv(x, 256, 3)  # skip connection
    x = MaxPool2D(2, 2, 'same')(x)
    x = DarknetConv(x, 512, 3)
    x = MaxPool2D(2, 1, 'same')(x)
    x = DarknetConv(x, 1024, 3)
    return tf.keras.Model(inputs, (x_8, x), name=name)


def YoloConv(filters, name=None):
    def yolo_conv(x_in):
        if isinstance(x_in, tuple):
            inputs = Input(x_in[0].shape[1:]), Input(x_in[1].shape[1:])
            x, x_skip = inputs

            # concat with skip connection
            x = DarknetConv(x, filters, 1)
            x = UpSampling2D(2)(x)
            x = Concatenate()([x, x_skip])
        else:
            x = inputs = Input(x_in.shape[1:])

        x = DarknetConv(x, filters, 1)
        x = DarknetConv(x, filters * 2, 3)
        x = DarknetConv(x, filters, 1)
        x = DarknetConv(x, filters * 2, 3)
        x = DarknetConv(x, filters, 1)
        return Model(inputs, x, name=name)(x_in)
    return yolo_conv


def YoloConvTiny(filters, name=None):
    def yolo_conv(x_in):
        if isinstance(x_in, tuple):
            inputs = Input(x_in[0].shape[1:]), Input(x_in[1].shape[1:])
            x, x_skip = inputs

            # concat with skip connection
            x = DarknetConv(x, filters, 1)
            x = UpSampling2D(2)(x)
            x = Concatenate()([x, x_skip])
        else:
            x = inputs = Input(x_in.shape[1:])
            x = DarknetConv(x, filters, 1)

        return Model(inputs, x, name=name)(x_in)
    return yolo_conv


def YoloOutput(filters, anchors, classes, name=None):
    def yolo_output(x_in):
        x = inputs = Input(x_in.shape[1:])
        x = DarknetConv(x, filters * 2, 3)
        x = DarknetConv(x, anchors * (classes + 5), 1, batch_norm=False)
        x = Lambda(lambda x: tf.reshape(x, (-1, tf.shape(x)[1], tf.shape(x)[2],
                                            anchors, classes + 5)))(x)
        return tf.keras.Model(inputs, x, name=name)(x_in)
    return yolo_output


def yolo_boxes(pred, anchors, classes):
    # pred: (batch_size, grid, grid, anchors, (x, y, w, h, obj, ...classes))
    grid_size = tf.shape(pred)[1:3]
    box_xy, box_wh, objectness, class_probs = tf.split(
        pred, (2, 2, 1, classes), axis=-1)
    # print(box_xy.shape)
    # print('#'*30)
    # print(box_wh.shape)
    box_xy = tf.sigmoid(box_xy)
    objectness = tf.sigmoid(objectness)
    class_probs = tf.sigmoid(class_probs)
    pred_box = tf.concat((box_xy, box_wh), axis=-1)  # original xywh for loss

    # !!! grid[x][y] == (y, x)
    grid = tf.meshgrid(tf.range(grid_size[1]), tf.range(grid_size[0]))
    grid = tf.expand_dims(tf.stack(grid, axis=-1), axis=2)  # [gx, gy, 1, 2]

    box_xy = (box_xy + tf.cast(grid, tf.float32)) / \
        tf.cast(grid_size, tf.float32)
    box_wh = tf.exp(box_wh) * anchors

    box_x1y1 = box_xy - box_wh / 2
    box_x2y2 = box_xy + box_wh / 2
    bbox = tf.concat([box_x1y1, box_x2y2], axis=-1)

    return bbox, objectness, class_probs, pred_box


def yolo_nms(outputs, anchors, masks, classes):
    # boxes, conf, type
    b, c, t = [], [], []

    for o in outputs:
        b.append(tf.reshape(o[0], (tf.shape(o[0])[0], -1, tf.shape(o[0])[-1])))
        c.append(tf.reshape(o[1], (tf.shape(o[1])[0], -1, tf.shape(o[1])[-1])))
        t.append(tf.reshape(o[2], (tf.shape(o[2])[0], -1, tf.shape(o[2])[-1])))

    bbox = tf.concat(b, axis=1)
    confidence = tf.concat(c, axis=1)
    class_probs = tf.concat(t, axis=1)

    scores = confidence * class_probs
    boxes, scores, classes, valid_detections = tf.image.combined_non_max_suppression(
        boxes=tf.reshape(bbox, (tf.shape(bbox)[0], -1, 1, 4)),
        scores=tf.reshape(
            scores, (tf.shape(scores)[0], -1, tf.shape(scores)[-1])),
        max_output_size_per_class=100,
        max_total_size=100,
        iou_threshold=0.4,
        score_threshold=0.35
    )

    return boxes, scores, classes, valid_detections


def YoloV3(size=None, channels=3, anchors=yolo_anchors,
           masks=yolo_anchor_masks, classes=3, training=True):

    x = inputs = tf.keras.layers.Input([size, size, channels], name='input')

    x_36, x_61, x = Darknet(name='yolo_darknet')(x)

    x = YoloConv(512, name='yolo_conv_0')(x)
    output_0 = YoloOutput(512, len(masks[0]), classes, name='yolo_output_0')(x)

    x = YoloConv(256, name='yolo_conv_1')((x, x_61))
    output_1 = YoloOutput(256, len(masks[1]), classes, name='yolo_output_1')(x)

    x = YoloConv(128, name='yolo_conv_2')((x, x_36))
    output_2 = YoloOutput(128, len(masks[2]), classes, name='yolo_output_2')(x)

    if training:
        return Model(inputs, (output_0, output_1, output_2), name='yolov3')

    boxes_0 = Lambda(lambda x: yolo_boxes(x, anchors[masks[0]], classes),
                     name='yolo_boxes_0')(output_0)
    boxes_1 = Lambda(lambda x: yolo_boxes(x, anchors[masks[1]], classes),
                     name='yolo_boxes_1')(output_1)
    boxes_2 = Lambda(lambda x: yolo_boxes(x, anchors[masks[2]], classes),
                     name='yolo_boxes_2')(output_2)

    outputs = Lambda(lambda x: yolo_nms(x, anchors, masks, classes),
                     name='yolo_nms')((boxes_0[:3], boxes_1[:3], boxes_2[:3]))

    return Model(inputs, outputs, name='yolov3')


def YoloV3Tiny(size=None, channels=3, anchors=yolo_tiny_anchors,
               masks=yolo_tiny_anchor_masks, classes=80, training=False):
    x = inputs = Input([size, size, channels], name='input')

    x_8, x = DarknetTiny(name='yolo_darknet')(x)

    x = YoloConvTiny(256, name='yolo_conv_0')(x)
    output_0 = YoloOutput(256, len(masks[0]), classes, name='yolo_output_0')(x)

    x = YoloConvTiny(128, name='yolo_conv_1')((x, x_8))
    output_1 = YoloOutput(128, len(masks[1]), classes, name='yolo_output_1')(x)

    if training:
        return Model(inputs, (output_0, output_1), name='yolov3')

    boxes_0 = Lambda(lambda x: yolo_boxes(x, anchors[masks[0]], classes),
                     name='yolo_boxes_0')(output_0)
    boxes_1 = Lambda(lambda x: yolo_boxes(x, anchors[masks[1]], classes),
                     name='yolo_boxes_1')(output_1)
    outputs = Lambda(lambda x: yolo_nms(x, anchors, masks, classes),
                     name='yolo_nms')((boxes_0[:3], boxes_1[:3]))
    return Model(inputs, outputs, name='yolov3_tiny')


def YoloLoss(anchors, classes=80, ignore_thresh=0.5):
    def yolo_loss(y_true, y_pred):
        # 1. transform all pred outputs
        # y_pred: (batch_size, grid, grid, anchors, (x, y, w, h, obj, ...cls))
        pred_box, pred_obj, pred_class, pred_xywh = yolo_boxes(
            y_pred, anchors, classes)
        pred_xy = pred_xywh[..., 0:2]
        pred_wh = pred_xywh[..., 2:4]

        # 2. transform all true outputs
        # y_true: (batch_size, grid, grid, anchors, (x1, y1, x2, y2, obj, cls))
        true_box, true_obj, true_class_idx = tf.split(
            y_true, (4, 1, 1), axis=-1)
        true_xy = (true_box[..., 0:2] + true_box[..., 2:4]) / 2
        true_wh = true_box[..., 2:4] - true_box[..., 0:2]

        # give higher weights to small boxes
        box_loss_scale = 2 - true_wh[..., 0] * true_wh[..., 1]

        # 3. inverting the pred box equations
        grid_size = tf.shape(y_true)[1]
        grid = tf.meshgrid(tf.range(grid_size), tf.range(grid_size))
        grid = tf.expand_dims(tf.stack(grid, axis=-1), axis=2)
        true_xy = true_xy * tf.cast(grid_size, tf.float32) - \
            tf.cast(grid, tf.float32)
        true_wh = tf.math.log(true_wh / anchors)
        true_wh = tf.where(tf.math.is_inf(true_wh),
                           tf.zeros_like(true_wh), true_wh)

        # 4. calculate all masks
        obj_mask = tf.squeeze(true_obj, -1)
        # ignore false positive when iou is over threshold
        best_iou = tf.map_fn(
            lambda x: tf.reduce_max(broadcast_iou(x[0], tf.boolean_mask(
                x[1], tf.cast(x[2], tf.bool))), axis=-1),
            (pred_box, true_box, obj_mask),
            tf.float32)
        ignore_mask = tf.cast(best_iou < ignore_thresh, tf.float32)

        # 5. calculate all losses
        xy_loss = obj_mask * box_loss_scale * \
            tf.reduce_sum(tf.square(true_xy - pred_xy), axis=-1)
        wh_loss = obj_mask * box_loss_scale * \
            tf.reduce_sum(tf.square(true_wh - pred_wh), axis=-1)
        obj_loss = binary_crossentropy(true_obj, pred_obj)
        obj_loss = obj_mask * obj_loss + \
            (1 - obj_mask) * ignore_mask * obj_loss
        # TODO: use binary_crossentropy instead
        class_loss = obj_mask * sparse_categorical_crossentropy(
            true_class_idx, pred_class)

        # 6. sum over (batch, gridx, gridy, anchors) => (batch, 1)
        xy_loss = tf.reduce_sum(xy_loss, axis=(1, 2, 3))
        wh_loss = tf.reduce_sum(wh_loss, axis=(1, 2, 3))
        obj_loss = tf.reduce_sum(obj_loss, axis=(1, 2, 3))
        class_loss = tf.reduce_sum(class_loss, axis=(1, 2, 3))

        return xy_loss + wh_loss + obj_loss + class_loss
    return yolo_loss


In [ ]:
model= YoloV3(size=416,channels=3,anchors=yolo_anchors,masks = yolo_anchor_masks,classes=3,training=True)

In [ ]:
# Training=True
model.summary()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Training = False. Used while testing the model.
model.summary()

In [ ]:
# Training = True
from tensorflow.keras.utils import plot_model
plot_model(model,show_shapes=True)

In [ ]:
# Training = False. Used while testing the model.
from tensorflow.keras.utils import plot_model
plot_model(model,show_shapes=True)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
from tensorflow.keras.callbacks import ReduceLROnPlateau,EarlyStopping,ModelCheckpoint,TensorBoard


"""

  Before Executing this snippet you have to create a .txt file and put all the names of the object classes you want
  Name that file as voc2012.names. or else change the path to your file in the snippet below.

"""
    

    
# model = YoloV3(416, training=True, classes=3)
anchors = yolo_anchors
anchor_masks = yolo_anchor_masks


train_dataset = load_tfrecord_dataset(
    '/content/data/OID_train.tfrecord','/content/data/voc2012.names' , 416)

train_dataset = train_dataset.shuffle(buffer_size=100)
train_dataset = train_dataset.batch(8)
train_dataset = train_dataset.map(lambda x, y: (
    transform_images(x, 416),
    transform_targets(y, anchors, anchor_masks,416)))
train_dataset = train_dataset.prefetch(
    buffer_size=tf.data.experimental.AUTOTUNE)

val_dataset = load_tfrecord_dataset(
    '/content/data/OID_validation.tfrecord','/content/data/voc2012.names',416 )

val_dataset = val_dataset.batch(8)
val_dataset = val_dataset.map(lambda x, y: (
    transform_images(x, 416),
    transform_targets(y, anchors, anchor_masks, 416)))


optimizer = tf.keras.optimizers.Adam(lr=0.001)
loss = [YoloLoss(anchors[mask], classes=3)
        for mask in anchor_masks]


model.compile(optimizer=optimizer, loss=loss,)

callbacks = [
    ReduceLROnPlateau(verbose=1),
    EarlyStopping(patience=2, verbose=1),
    # ModelCheckpoint('/content/drive/MyDrive/OD/checkpoint',
    ModelCheckpoint('/content/model_saved/checkpoint',
                    verbose=1, save_weights_only=True,save_best_only=True),
    TensorBoard(log_dir='logs')
]

history = model.fit(train_dataset,
                    epochs=10,
                    callbacks=callbacks,
                    validation_data=val_dataset)

In [ ]:
"""I am not retraining the network. Just loading my trained  network weights here"""
model.load_weights('/content/drive/MyDrive/Yolov3/checkpoint')

In [ ]:
"""This is how my tensorboard plot looked like after training. As you can see the loss decreases as the number of epochs increased."""
%load_ext tensorboard
%tensorboard --logdir /content/yolov3-tf2/logs

# **Prediction Using My Trained Network Weights**

In these whole markdown cells for predictions you can observe that model is able to find the object in the image but the boudning box size is constant. That is because I have trained my model on the random images from Open Image v6. This is why that problem of same sized Bounding Box is coming up. 

In [ ]:
import time
import cv2
import numpy as np
import tensorflow as tf
from google.colab.patches import cv2_imshow
yolo = YoloV3(classes=3,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3/checkpoint').expect_partial()
# class_names = [c.strip() for c in open('/content/data/voc2012.names','r').readlines()]
class_names=[]
for c in open('/content/data/voc2012.names').readlines():
  class_names.append(c)
img_raw = tf.image.decode_image(open('/content/OIDv4_ToolKit/OID/Dataset/validation/Bus/282e0c8f8ef8ba8d.jpg', 'rb').read(), channels=3) # worked
img = tf.expand_dims(img_raw, 0)
img = transform_images(img, 416)
boxes, scores, classes, nums = yolo(img)
img = cv2.cvtColor(img_raw.numpy(), cv2.COLOR_RGB2BGR)
img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
cv2_imshow(img)
cv2.imwrite('/content/output.jpg', img)

In [ ]:
import time
import cv2
import numpy as np
import tensorflow as tf
yolo = YoloV3(classes=3,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3/checkpoint').expect_partial()
# class_names = [c.strip() for c in open('/content/data/voc2012.names','r').readlines()]
class_names=[]
for c in open('/content/data/voc2012.names').readlines():
  class_names.append(c)
img_raw = tf.image.decode_image(open('/content/OIDv4_ToolKit/OID/Dataset/train/Car/eed5a4944f9206ce.jpg', 'rb').read(), channels=3) # worked
img = tf.expand_dims(img_raw, 0)
img = transform_images(img, 416)
boxes, scores, classes, nums = yolo(img)
img = cv2.cvtColor(img_raw.numpy(), cv2.COLOR_RGB2BGR)
img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
cv2.imwrite('/content/output.jpg', img)

In [ ]:
os.chdir('/content/yolov3-tf2')

In [ ]:
import cv2
vidcap = cv2.VideoCapture('/content/sub-1504614469486.mp4')
def getFrame(sec):
    vidcap.set(cv2.CAP_PROP_POS_MSEC,sec*1000)
    hasFrames,image = vidcap.read()
    if hasFrames:
        cv2.imwrite("image"+str(count)+".jpg", image)     # save frame as JPG file
    return hasFrames
sec = 0
frameRate = 0.5 #//it will capture image in each 0.5 second
count=1
success = getFrame(sec)
while success:
    count = count + 1
    sec = sec + frameRate
    sec = round(sec, 2)
    success = getFrame(sec)

In [ ]:
yolo = YoloV3(classes=3,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3/checkpoint').expect_partial()
class_names=[]
for c in open('/content/data/voc2012.names').readlines():
  class_names.append(c)
img_raw = tf.image.decode_image(open('/content/OIDv4_ToolKit/OID/Dataset/train/Truck/00e575172c598e71.jpg', 'rb').read(), channels=3) 
img = tf.expand_dims(img_raw, 0)
img = transform_images(img, 416)
boxes, scores, classes, nums = yolo(img)
img = cv2.cvtColor(img_raw.numpy(), cv2.COLOR_RGB2BGR)
img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
cv2_imshow(img)
cv2.imwrite('/content/output.jpg', img)



# **Predictions Using The Pretrained YOLOv3 Weights**

Since I was getting problem with the bounding box size and predictions I chose to use the pretrained model rather than training from scratch. So in these markdown cells I am using pretrained YOLOv3 weights.

In [ ]:
!wget https://pjreddie.com/media/files/yolov3.weights -O data/yolov3.weights

In [ ]:
yolo = YoloV3(classes=80)
yolo.summary()
load_darknet_weights(yolo, '/content/yolov3-tf2/data/yolov3.weights', False)
img = np.random.random((1, 320, 320, 3)).astype(np.float32)
output = yolo(img)
yolo.save_weights('/content/drive/MyDrive/Yolov3_Pretrained/checkpoint')

Here we can see that after using the pretrained weights I am getting good results with the bounnding boxes, ie. Bounding boxes are exactly of the sizes of the vehicle present unlike the previous stage.

In [ ]:
yolo = YoloV3(classes=80,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3_Pretrained/checkpoint').expect_partial()
class_names=[]
for c in open('/content/yolov3-tf2/data/coco.names').readlines():
  class_names.append(c)
img_raw = tf.image.decode_image(open('/content/OIDv4_ToolKit/OID/Dataset/validation/Truck/4f7db69adf060f1b.jpg', 'rb').read(), channels=3) # worked
img = tf.expand_dims(img_raw, 0)
img = transform_images(img, 416)
boxes, scores, classes, nums = yolo(img)
img = cv2.cvtColor(img_raw.numpy(), cv2.COLOR_RGB2BGR)
img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
cv2_imshow(img)
cv2.imwrite('/content/output.jpg', img)

# **Predictions On Video Using My Trained Network**

In [ ]:
!git clone https://github.com/zzh8829/yolov3-tf2.git

In [ ]:
import time
from google.colab.patches import cv2_imshow
out = '/content/output_video'
yolo = YoloV3(classes=3,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3/checkpoint')
class_names = [c.strip() for c in open('/content/yolov3-tf2/data/voc2012.names').readlines()]

times = []

output_format = 'X264'
try:
    vid = cv2.VideoCapture(int('/content/sub-1504614469486.mp4'))
except:
    vid = cv2.VideoCapture('/content/sub-1504614469486.mp4')


# if out:
    # by default VideoCapture returns float instead of int
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))
codec = cv2.VideoWriter_fourcc(*output_format)
writer = cv2.VideoWriter("output.mp4",cv2.VideoWriter_fourcc(*"XVID"), 30,(640,480))
a = 0
while True:
    _, img = vid.read()
    a+=1

    img_in = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    
    img_in = tf.expand_dims(img_in, 0)
    img_in = transform_images(img_in, 416)

    t1 = time.time()
    boxes, scores, classes, nums = yolo.predict(img_in)
    t2 = time.time()
    times.append(t2-t1)
    times = times[-20:]

    img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
    # img = cv2.putText(img, "Time: {:.2f}ms".format(sum(times)/len(times)*1000), (0, 30),
    #                   cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 0, 255), 2)

    # out.write(img)
    cv2_imshow(img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if a==85:
      break
vid.release()
# out.release()
cv2.destroyAllWindows()


# **Predictions on Video Using YOLOv3 Pretrained Weights**

In [ ]:
import time
from google.colab.patches import cv2_imshow
out = '/content/output_video'
yolo = YoloV3(classes=80,training=False)
yolo.load_weights('/content/drive/MyDrive/Yolov3_Pretrained/checkpoint')
class_names = [c.strip() for c in open('/content/yolov3-tf2/data/coco.names').readlines()]
times = []
output_format = 'X264'
try:
    vid = cv2.VideoCapture(int('/content/sub-1504614469486.mp4'))
except:
    vid = cv2.VideoCapture('/content/sub-1504614469486.mp4')
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))
codec = cv2.VideoWriter_fourcc(*output_format)
out = cv2.VideoWriter('output_video.avi', codec, fps, (width, height))

while True:
    _, img = vid.read()


    img_in = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    
    img_in = tf.expand_dims(img_in, 0)
    img_in = transform_images(img_in, 416)

    t1 = time.time()
    boxes, scores, classes, nums = yolo.predict(img_in)
    t2 = time.time()
    times.append(t2-t1)
    times = times[-20:]

    img = draw_outputs(img, (boxes, scores, classes, nums), class_names)
    out.write(img)
    cv2_imshow(img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

# **Observations**



*   We can observe that the my trained network fails to detect objects in every frame. It detects the object in some of the images. But the bounding boxes around the objects are of same size which is not desirable in the object tracking problem which we are going to try further. This problem is arising because I am using image data from Open Image v6. It randomly selects the images from its dataset. So the images I trained on were very different in sizes or differnet form. The problem might be arising due to this. I have used 1000 images per class while training network.
*   We can see that the YOLOv3 with pretrained weights performs well for detection. That is because YOLOv3 was pretrained on Imagenet Dataset which is alltogether a huge dataset to train on.



# **Conclusion**



*   Since I am going to try the object tracking by using YOLOv3 for object detection I am going to use pretrained network for detection since it will give me better result.



# References



*   https://pjreddie.com/media/files/papers/YOLOv3.pdf
*   https://github.com/pythonlessons/TensorFlow-2.x-YOLOv3
*   https://github.com/zzh8829/yolov3-tf2




